# Consolidation Exercise 2 — Comparing Algorithms

So far, we've covered:

+ Variables, data types and operators
+ If-then-else
+ Loops (for, while)
+ Functions
+ Lists
+ Input and Output

This week, we'll be doing our second **consolidation** exercise. As a reminder, our aim here is to give you a less structured exercise, closer to what you'll face after this course, and as preparation for the summative assessment at the end of the course. Before diving in and writing code, you should think carefully about how you can break the larger task down into smaller chunks, and how you can use functions to structure and organise your code. As before, we'll provide formative feedback for work submitted via Blackboard. The due date date for this submission is 29th November at 5pm. 


## Introduction

Engineers typically compare the complexity of off-the-shelf algorithms using a combination of:
- theoretical analysis using (Big O)
- measuring actual performance through experiments (benchmarking)
- practical resource requirements (real contraints and data)

The important distinction is that Big O notation tells you how an algorithm scales, but it doesn't necessarily tell you which algorithm will perform best in a real engineering application.

## Task

You are developing an autonomous mobile robot for use in a warehouse. 

The robot needs to recognise obstacles using an array of distance sensors. 

A simple array could include 3 distance sensors and record three distance measurements per observation. 

In contrast, a depth camera is equivalent to multiple distance sensors, and could record hundereds to thousands of distance measurements per observation.

<img src="https://github.com/engmaths/SEMT10002_2026/blob/main/media/week_10/robots_with_sensors.png?raw=true" width="60%">

To classify a new observation, the observation should be compared with some existing data that has already been classified. This data, labelled with the correct classification is called the training data. 

Here is an example for a robot with 3 sensors:

| Left (cm) | Centre (cm) | Right (cm) | Classification |
|-----------|-------------|------------|----------------|
| 100 | 100 | 100 | Clear |
| 80 | 90 | 85 | Clear |
| 20 | 100 | 100 | Obstacle left |
| 15 | 30 | 100 | Obstacle left |
| 100 | 20 | 100 | Obstacle ahead |
| 100 | 15 | 25 | Obstacle ahead |
| 100 | 100 | 20 | Obstacle right |
| 90 | 100 | 15 | Obstacle right |
| 10 | 15 | 20 | Surrounded |

The input size is $n$ observations $\times$ $m$ sensors 

This exercise has three parts in which you will compare algorithms for recognising and classifying obstacles using:
1. **Algorithmic complexity**: How does the computational effort grow as the input size increases?
2. **Computational performance (benchmarking)**: How long does the algorithm actually take to execute, and how much memory does it use?
3. **Engineering suitability**: Is the algorithm appropriate for the constraints of the application?

# K-nearest neighbours (k-NN) 

k-NN is a simple machine learning algorithm used to classify new observations based on previously collected data.

For example, an autonomous robot might use sensor measurements to recognise objects. It has a dataset of previously observed objects, each labelled with a category such as box, person or pallet. The algorithm is called k-nearest neighbours because it uses the k closest observations to make its prediction.

When the robot encounters a new object, k-NN:
1. Calculates the distance between the new observation and every observation in its training dataset.
2. Identifies the k most similar (nearest) observations.
3. Predicts the category by selecting the most common label among those neighbours.

<img src="https://github.com/engmaths/SEMT10002_2026/blob/main/media/week_10/KNN.png?raw=true" width="50%">

Below are two different implementations of kNN that produce the same predictions but use different approaches to identify the nearest neighbours.

### k-NN A

This implementation:
- A. Calculates distance from the new observation to every training observation
- B. Sorts all the distances using the bubble sort algorithm
- C. Counts the labels of the k nearest neighbours

Bubble sort repeatedly compares adjacent elements and swaps them if they are in the wrong order.

<img src="https://github.com/engmaths/SEMT10002_2026/blob/main/media/week_10/bubble_sort.png?raw=true" width="35%">


In [ ]:
def knn_A(training_data, new_observation, k):
    """
    Calculates distance from the new observation to every training observation
    Sorts all the distances using bubble sort
    Counts the labels of the k nearest neighbours
    Returns the most common label
    """
    distances = []

    # Calculate distance to every training observation
    for features, label in training_data:
        distance = 0
        for i in range(len(new_observation)):
            distance += (features[i] - new_observation[i])**2
        distances.append((distance, label))

    # Sort all distances using bubble sort
    n = len(distances)
    for i in range(n):
        for j in range(0, n-i-1):
            if distances[j][0] > distances[j+1][0]:
                distances[j], distances[j+1] = distances[j+1], distances[j]

    # Count the labels of the k nearest neighbours
    votes = {}
    for i in range(k):
        label = distances[i][1]
        votes[label] = votes.get(label, 0) + 1

    # Return the most common label
    return max(votes, key=votes.get)

### KNN B 
This implementation:
- A. Calculate distance from the new observation to every training observation
- B. Maintains a sorted list containing only the k closest observations.
- C. Counts the labels of the k nearest neighbours

Insertion sort repeatedly takes takes the first item from the unsorted section of a list and compares this item *backwards* against the sorted section of the list, shifting larger elements to the right.

<img src="https://github.com/engmaths/SEMT10002_2026/blob/main/media/week_9/insertion_sort.png?raw=true" width="60%">

In [ ]:
def knn_B(training_data, new_observation, k):

    """
    Maintains a sorted list containing only the k closest observations
    """

    nearest = []

    for features, label in training_data:

        # Step A: Calculate distance to every training observation
        distance = 0
        for i in range(len(new_observation)):
            distance += (features[i] - new_observation[i])**2

        # Step B: Maintains a sorted list containing only the k closest observations.
        # Add the new observation to the end of the list of closest observations
        nearest.append((distance, label))

        # Gets the index of the new observation in the list
        index = len(nearest) - 1

        # Use insertion sort (for final list element only) to move it backwards to the correct position
        while index > 0 and nearest[index-1][0] > nearest[index][0]:
            nearest[index], nearest[index-1] = nearest[index-1], nearest[index]
            index -= 1

        # Keep only the k nearest neighbours
        if len(nearest) > k:
            nearest.pop()

    # Step C: Count the labels of the k nearest neighbours
    votes = {}
    for distance, label in nearest:
        votes[label] = votes.get(label, 0) + 1

    # Return the most common label
    return max(votes, key=votes.get)
    

The following example tests both functions with a smal sample data set - each produces the correct result


In [23]:
sample_data = [
    ([1, 2], "cat"),
    ([2, 3], "cat"),
    ([3, 3], "cat"),
    ([6, 7], "dog"),
    ([7, 8], "dog"),
    ([8, 7], "dog"),
    ([2, 6], "cat"),
    ([7, 3], "dog")
]

new_observation = [3, 4]

print(knn_A(sample_data, new_observation, 3))

print(knn_B(sample_data, new_observation, 3))

cat
cat


## Part 1 - Algorithmic Complexity

Analyse the complexity of each algorithm...
1. Analyse complexity of each step within the algorithm individually (calculate distances, sort results, store data)
2. Add complexity of sequential operations 
3. Simplify to the dominant term to get the complexity of the algorithm

***

### Example: Complexity of k-NN B

### 1. Analyse complexity of each step within the algorithm

##### **Step A:**
```Python
#  Step A: Calculate distance to every training observation
for features, label in training_data:

    distance = 0

    for i in range(len(new_observation)):
        distance += (features[i] - new_observation[i])**2
```
**Time complexity is $O(nm)$**: Grows linearly with number of observations $n$ (outer loop), and grows linearly with number of senosrs $m$ (inner loop)<br>
 **Space complexity is $O(1)$**: Size of `distance` is constant for any number of observations $n$

##### **Step B:**
```Python
nearest = []

for features, label in training_data:

    ...

    # Step B: Maintains a sorted list containing only the k closest observations.
    # Add the new observation to the end of the list of closest observations
    nearest.append((distance, label))

    # Gets the index of the new observation in the list
    index = len(nearest) - 1

    # Use insertion sort (for final list element only) to move it backwards to the correct position
    while index > 0 and nearest[index-1][0] > nearest[index][0]:
        nearest[index], nearest[index-1] = nearest[index-1], nearest[index]
        index -= 1

    # Keep only the k nearest neighbours
    if len(nearest) > k:
        nearest.pop()
```
**Time complexity is $O(n)$**: Grows linearly with number of observations $n$ (outer loop). <br>Inner while loop has constant complexity for any value of $n$ and $m$ (*but* grows linearly with k (number of nearest neighbours used))

**Space complexity is $O(1)$**: Constant complexity for any value of $n$ and $m$ <br>(*but* `nearest` grows linearly with k (number of nearest neighbours used))

##### **Step C:**
```Python
# Step C: Count the labels of the k nearest neighbours
votes = {}
for distance, label in nearest:
    votes[label] = votes.get(label, 0) + 1

# Return the most common label
return max(votes, key=votes.get)
```
**Time complexity is $O(1)$**: Constant for any value of $n$ and $m$ <br>(*but* grows linearly with k (number of nearest neighbours used))

**Space complexity is $O(1)$**: Constant for any value of $n$ and $m$ <br>(*but* grows linearly with k (number of nearest neighbours used))

***

### 2. Add complexity of sequential operations to get complexity of the algorithm

**Time complexity** = $\underbrace{O(nm)}_{\text{Step A}} + \underbrace{O(n)}_{\text{Step B}} + \underbrace{O(1)}_{\text{Step C}}$

**Space complexity** = $\underbrace{O(1)}_{\text{Step A}} + \underbrace{O(1)}_{\text{Step B}} + \underbrace{O(1)}_{\text{Step C}}$
***

### 3. Simplify to the dominant term

**Time complexity of the algorithm** = $O(n^2) + O(nm)$ <br>
($O(n^2)$ is the highest order $n$ term and $O(nm)$ is the highest order $m$ term)

**Space complexity of the algorithm** = $O(1)$<br>
($O(1)$ is the highest order term

***

### Example: Complexity of k-NN A

Solve this for each step of the algorithm and note your answers down in `README.txt`

### 1. Analyse complexity of each step within the algorithm

##### **Step A:**
```Python
distances = []

#  Step A: Calculate distance to every training observation
for features, label in training_data:

    distance = 0

    for i in range(len(new_observation)):
        distance += (features[i] - new_observation[i])**2
    distances.append((distance, label))
```
**Time complexity is $O(nm)$**: Grows linearly with number of observations $n$ (outer loop), and grows linearly with number of sensors $m$ (inner loop)<br>
 **Space complexity is $O(n)$**: `distances` grows linearly with number of observations $n$





##### **Step B:**
```Python
# Step B: Sort all distances using bubble sort
n = len(distances)
for i in range(n):
    for j in range(0, n-i-1):
        if distances[j][0] > distances[j+1][0]:
            distances[j], distances[j+1] = distances[j+1], distances[j]
```
**Time complexity is $O(n^2)$**: Grows quadratically with number of observations $n$ (number of comparisons is $k = \frac{n(n-1)}{2}$ - same as finding repeats example from Algorithmic Complexity seminar)<br>
**Space complexity is $O(1)$**: No additional variables created

##### **Step C:**
```Python
# Step C: Count the labels of the k nearest neighbours
votes = {}
for i in range(k):
    label = distances[i][1]
    votes[label] = votes.get(label, 0) + 1

# Return the most common label
return max(votes, key=votes.get)
```
**Time complexity is $O(1)$**: Constant for any value of $n$ and $m$ <br>(*but* grows linearly with k (number of nearest neighbours used))<br>

**Space complexity is $O(1)$**: Constant for any value of $n$ and $m$ <br>(*but* grows linearly with k (number of nearest neighbours used))
***




### 2. Add complexity of sequential operations to get complexity of the algorithm*

**Time complexity** = $\underbrace{O(nm)}_{\text{Step A}} + \underbrace{O(n^2)}_{\text{Step B}} + \underbrace{O(1)}_{\text{Step C}}$

**Space complexity** = $\underbrace{O(n)}_{\text{Step A}} + \underbrace{O(1)}_{\text{Step B}} + \underbrace{O(1)}_{\text{Step C}}$


***

### 3. Simplify to the dominant term

**Time complexity of the algorithm** = $O(n^2) + O(nm)$ <br>
($O(n^2)$ is the highest order $n$ term and $O(nm)$ is the highest order $m$ term)

**Space complexity of the algorithm*** = $O(n)$<br>
($O(n)$ is the highest order $n$ term 

***

## Part 2 - Computational performance (benchmarking)

The file `obstacle_avoidance_data.csv` contains 20,000 labelled observations from three distance sensors (left, centre, right).

| Left (cm) | Centre (cm) | Right (cm) | Classification |
|-----------|-------------|------------|----------------|
| 100 | 100 | 100 | Clear |
| 80 | 90 | 85 | Clear |
| 20 | 100 | 100 | Obstacle left |
| 15 | 30 | 100 | Obstacle left |
| 100 | 20 | 100 | Obstacle ahead |
| 100 | 15 | 25 | Obstacle ahead |
| 100 | 100 | 20 | Obstacle right |
| 90 | 100 | 15 | Obstacle right |
| 10 | 15 | 20 | Surrounded |

Your task is to compare their computational efficiency as the training dataset grows. Compare the computational performance (execution time and memory usage) of the two algorithms as the size of the input data (number of observations) increases:
- Import data from files in the folder knn_datasets
- Start by using k=5 nearest neightbours
- Create a value for a new observation (e.g. generate a random value)
- Measure execution time in seconds with `time.perf_counter()` (example below)
- Measure memory use with `tracemalloc.start()` (example below)
    - Perform the two measurements separately, and exclude print statements, to avoid distorting the measurements - use command line arguments and flags to configure your program to operate in these different modes 
    - Repeat measurements at least three times and calculate the average (execution time can vary due to other processes running on the computer, background activity and other sources of measurement noise).
    - Perform experiments on the same computer as the computer you use effects the execultion time 
- Record your results to an output file. How do they compare with the predicted time and space complexity (Big O) of the algorithm? 
    - Does increasing $n$ affect both implementations in the same way?
    - Do the experimental results agree with your Big O analysis?
    - Does changing the value of k effect the memory usage or execution time? Why do you think this is? (Try k = 50, k = 500 ...)
    - Does the distribution of distances (ascending, descending, random), affect memory usage or execution time?

The following example shows how to time the execution of a section of code:

In [29]:
import time 

start_time = time.perf_counter()

knn_A(sample_data, new_observation, 3)

end_time = time.perf_counter()

execution_time = end_time - start_time
print(f'Algorithm A took {execution_time} seconds' )

Algorithm A took 0.00010414503049105406 seconds


The following example shows how to measure the space taken by a secion of code:

In [30]:
import tracemalloc

tracemalloc.start()

knn_A(sample_data, new_observation, 3)

current, peak = tracemalloc.get_traced_memory()

tracemalloc.stop()

print(f'Peak memory usage = {peak / 1024} KB')

Peak memory usage = 18.7197265625 KB


***

## Answer

| Operation                         | Implementation A | Implementation B |
|-----------------------------------|------------------|------------------|
| Calculate distances               | \(O(n)\)         | \(O(n)\)         |
| Sort / maintain nearest neighbours | \(O(n^2)\)       | \(O(nk)\)        |
| Additional storage                | \(O(n)\)         | \(O(k)\)         |
| Overall time (fixed \(k\))        | \(O(n^2)\)       | \(O(n)\)         |

There are several important factors:
1. Number of training observations (\(n\)): As the dataset grows, the difference between \(O(n^2)\) and \(O(nk)\) becomes increasingly important.
2. Number of neighbours (\(k\)): If \(k\) is small relative to \(n\), maintaining a sorted list of only the nearest \(k\) observations is likely to be much more efficient than sorting all \(n\) observations.
3. Number of features (\(d\)): Calculating each distance takes \(O(d)\), so the distance calculation actually takes \(O(nd)\). This affects both implementations.
4. Distribution of distances: This can affect the number of comparisons needed to find the insertion position in Implementation B. - The position at which a new observation is inserted depends on its distance relative to the observations already stored.
- New distances are mostly larger: observations are usually inserted at the end. Fewer comparisons and less shifting of list elements are needed.
- New distances are mostly smaller: observations are inserted at the beginning. More comparisons and more shifting of list elements are needed.
- Distances are randomly distributed: insertion positions vary.

| Factor                    | Implementation A (sort all)          | Implementation B (maintain top k) |
|---------------------------|--------------------------------------|-----------------------------------|
| Small dataset             | Difference may be negligible         | Difference may be negligible      |
| Large dataset, small \(k\) | More work sorting unnecessary observations | Likely preferable            |
| Large \(k\), close to \(n\) | May be preferable                   | Less advantage                    |
| Distribution of distances | Affects sorting behaviour             | Affects insertion comparisons and shifting |

***

## Part 2 - Navigating a warehouse

<img src="https://github.com/engmaths/SEMT10002_2026/blob/main/media/week_10/warehouse.jpeg?raw=true" width="55%">


Imagine a robot entering a warehouse through the main entrance. It needs to explore all the locations to build a map.

The warehouse can be represented as a tree structure (acyclic graph). 

Depth-first-search and breadth-first-search algorithms visit every location in the warehouse. 

1. What is are the time and space complexity (Big O classification) of each algorithm? 
    - What is the Big O classification of each key step within the algorithm? - analyse these individually (calculate distances, sort results, store data)
    - Add sequential operations. If operations happen one after another, their complexities are added.
    - Simplify to the dominant term. When expressing overall Big O, keep the term that grows fastest.
    - Compare the algorithms. Which has greater time complexity? 

2. Compare the computational performance (execution time and memory usage) of the two algorithms as the size of the input data (number of observations) increases? 
- Use the `warehouse_generator` function with adjustable parameters, to create different tree structures and observe how the algorithms behave.
    - depth: Number of levels in the warehouse
    - branching: Number of locations connected to each location
    - target_depth: How far away the target location is
- Record your results to an output file. How do they compare with the predicted time and space complexity (Big O) of the algorithm? 
    - Does increasing $n$ affect both implementations in the same way?
    - Do the experimental results agree with your Big O analysis?
    - Is the performance affected by cycles in the network (Try: Office → Reception → Office; Storage A → Aisle A1 → Shelf A1-2 → Storage A)

- Example Cases:
    - Warehouse A: branching = 2, depth = 8
    - Warehouse B: branching = 8, depth = 3
    - Warehouse C: branching = 3, depth = 5





In [ ]:
warehouse = {
    "Entrance": ["Reception", "Storage A", "Storage B", "Loading Bay"],
    "Reception": ["Office", "Control Room"],
    "Office": [],
    "Control Room": [],
    "Storage A": ["Aisle A1", "Aisle A2"],
    "Aisle A1": ["Shelf A1-1", "Shelf A1-2"],
    "Shelf A1-1": [],
    "Shelf A1-2": [],
    "Aisle A2": ["Shelf A2-1", "Shelf A2-2"],
    "Shelf A2-1": [],
    "Shelf A2-2": [],
    "Storage B": ["Aisle B1"],
    "Aisle B1": ["Shelf B1-1", "Shelf B1-2"],
    "Shelf B1-1": [],
    "Shelf B1-2": [],
    "Loading Bay": ["Dispatch", "Receiving"],
    "Dispatch": [],
    "Receiving": []
}

In [ ]:
def bfs(warehouse, start):

    """
    BFS - explore level by level
    """

    queue = [start]
    visited = []

    while queue:

        current = queue.pop(0)
        visited.append(current)

        for neighbour in warehouse[current]:
            queue.append(neighbour)

    return visited

In [ ]:
def dfs(warehouse, start):
    """
    DFS – explore one branch as deeply as possible
    """

    stack = [start]
    visited = []

    while stack:

        current = stack.pop()
        visited.append(current)

        for neighbour in reversed(warehouse[current]):
            stack.append(neighbour)

    return visited

In [ ]:
def make_warehouse(branching, depth):

    """
    Generates a tree automatically, so students can change the parameters 
    without having to construct the warehouse manually.
    """

    warehouse = {"Entrance": []}
    current_level = ["Entrance"]
    counter = 0

    for level in range(depth):

        next_level = []

        for location in current_level:

            for i in range(branching):

                counter += 1
                new_location = f"Level{level+1}_{counter}"

                warehouse[location].append(new_location)
                warehouse[new_location] = []

                next_level.append(new_location)

        current_level = next_level

    return warehouse

warehouse = make_warehouse(branching=3, depth=4)

print("Number of locations:", len(warehouse))

Give students these versions, which record the maximum number of locations waiting to be explored at any point.

Important teaching distinction: these functions also store every visited location in a list. The max_memory measurement deliberately measures only the search frontier (the queue or stack), not the complete memory footprint of the program. Otherwise, storing the visited list would obscure the difference we're trying to demonstrate.

In [ ]:
def bfs(warehouse, start):

    queue = [start]
    visited = []
    max_memory = 0

    while queue:

        max_memory = max(max_memory, len(queue))

        current = queue.pop(0)
        visited.append(current)

        for neighbour in warehouse[current]:
            queue.append(neighbour)

    return visited, max_memory

def dfs(warehouse, start):

    stack = [start]
    visited = []
    max_memory = 0

    while stack:

        max_memory = max(max_memory, len(stack))

        current = stack.pop()
        visited.append(current)

        for neighbour in reversed(warehouse[current]):
            stack.append(neighbour)

    return visited, max_memory

warehouse = make_warehouse(branching=3, depth=4)

bfs_result, bfs_memory = bfs(warehouse, "Entrance")
dfs_result, dfs_memory = dfs(warehouse, "Entrance")

print("Number of locations:", len(warehouse))
print("BFS maximum queue size:", bfs_memory)
print("DFS maximum stack size:", dfs_memory)

Investigate how the shape of a warehouse affects the memory requirements of BFS and DFS. You can change the number of branches and the depth of the warehouse. Identify a configuration where BFS requires considerably more memory than DFS. Explain why.

suggested starting configurations:
Experiment	Branching	Depth	Warehouse shape
A	2	8	Deep and narrow
B	8	3	Wide and shallow
C	3	5	Balanced
D	5	5	Large and wide

record the maximum queue and stack sizes and explain the results.



Suppose the robot is looking for a particular storage location. Which algorithm finds it first?

In [5]:
'''
Cycles:
- Office → Reception → Office
- Storage A → Aisle A1 → Shelf A1-2 → Storage A
'''


warehouse = {
    "Entrance": ["Reception", "Storage A", "Storage B", "Loading Bay"],
    "Reception": ["Office", "Control Room"],
    "Office": ["Reception"],
    "Control Room": [],
    "Storage A": ["Aisle A1", "Aisle A2"],
    "Aisle A1": ["Shelf A1-1", "Shelf A1-2"],
    "Shelf A1-1": [],
    "Shelf A1-2": ["Storage A"],  # Loop back
    "Aisle A2": ["Shelf A2-1", "Shelf A2-2"],
    "Shelf A2-1": [],
    "Shelf A2-2": [],
    "Storage B": ["Aisle B1"],
    "Aisle B1": ["Shelf B1-1", "Shelf B1-2"],
    "Shelf B1-1": [],
    "Shelf B1-2": [],
    "Loading Bay": ["Dispatch", "Receiving"],
    "Dispatch": [],
    "Receiving": []
}



When the network contains loops, students must introduce a visited set.

In [6]:
visited = set()
queue = ["Entrance"]

while queue:
    current = queue.pop(0)

    if current not in visited:
        visited.add(current)
        print(current)

        for neighbour in warehouse[current]:
            if neighbour not in visited:
                queue.append(neighbour)

Entrance
Reception
Storage A
Storage B
Loading Bay
Office
Control Room
Aisle A1
Aisle A2
Aisle B1
Dispatch
Receiving
Shelf A1-1
Shelf A1-2
Shelf A2-1
Shelf A2-2
Shelf B1-1
Shelf B1-2


***
### Answer

This leads naturally into your intended comparison: both are \(O(n)\) time, but their space requirements depend on the shape of the tree.

There are two important things for them to discover:
1. Memory: BFS stores locations waiting to be explored, so it can use a lot of memory in a wide warehouse. DFS stores the current path and unexplored branches, so it generally uses less memory in a deep, narrow warehouse.
2. Finding the nearest location: BFS is the better choice when every connection represents an equal travel distance, because it explores locations in increasing order of distance from the entrance. DFS does not guarantee this.

***

## Getting started

We've provided you with a file `comparing_algorithms.py` to write your code in. There's a small amount of starter code here, which we've copied below- **Do not write your code for this exercise here!**

In [ ]:
import sys 

def calculate_mean(numbers):

    return 0.0

def calculate_median(numbers):

    return 0.0

def calculate_mode(numbers):

    return 0.0

def main(args):

    return

if __name__ == '__main__':
    
    #This line will run the main function of your code- uncomment this when you're ready to run everything.
	#main(sys.argv[1:])
     
    #The code below will run the functions for calculating each statistic
    #once these are done, you should comment these out and use the main function instead
    test_numbers = [1, 2, 3, 4, 5]
    print("Mean: ", calculate_mean(test_numbers))
    print("Median: ", calculate_median(test_numbers))
    print("Mode: ", calculate_mode(test_numbers))

On line 1, we import the *sys* module which you saw last week in the Input and Output lab. We'll use this to get access to the variables `sys.argv`. As a reminder  `sys.argv` (argument value) is a list containing any arguments entered on the command line. `sys.argv` will always contain at least one element as the name of the program we run counts as an argument. We've given you some code *cl_arguments.py* that you can run to see how this works.

On lines 3-14, we've defined some functions you to complete- one for each statistic to calculate.

In [ ]:
def calculate_mean(numbers):

    return 0.0

def calculate_median(numbers):

    return 0.0

def calculate_mode(numbers):

    return 0.0

On line 15, we've defined our *main* function. This function should be completed to draw all the parts of your program together- i.e it should read information from the command line (or file if necessary), pass it to the correct statistical functions and then print the output.

In [ ]:
def main(args):

    return

On lines 19-29, we've got some code that determines what happens when your code is run.

In [ ]:
if __name__ == '__main__':
    
    #This line will run the main function of your code- uncomment this when you're ready to run everything.
	#main(sys.argv[1:])
     
    #The code below will run the functions for calculating each statistic
    #once these are done, you should comment these out and use the main function instead
    test_numbers = [1, 2, 3, 4, 5]
    print("Mean: ", calculate_mean(test_numbers))
    print("Median: ", calculate_median(test_numbers))
    print("Mode: ", calculate_mode(test_numbers))


Currently, it will define a list `test_numbers` and call the statistical functions with `test_numbers` as input. We've put this here as we suggest that you start on this assigment by completing those three functions first. Once you've got them working, then you should work on your main function and will want to comment out these lines and uncomment the call to main on line 22. 

Please note that we are not suggesting these are the only functions you will need for this assigment; any time you can reduce repitition, or make code easier to read by using a function you should. 
